In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from marine_qc import (
    Climatology,
    do_bayesian_buddy_check,
)

In [ ]:
from data import get_climatology_data, get_grouped_data

# How to use quality control checks with grouped reports

We need some text!!!

In [ ]:
data = get_grouped_data()
data

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

colors = plt.cm.tab10.colors

i = 0
for color, (platform, group) in zip(colors, data.groupby("platform"), strict=False):
    ax1.plot(
        group["lon"],
        group["lat"],
        "-o",
        ms=5,
        label=platform,
        color=color,
    )

    ax2.plot(
        group["date"],
        group["sst"],
        "-o",
        ms=5,
        label=platform,
        color=color,
    )

    outlier = data.loc[(data.platform == platform) & (data.date == pd.Timestamp(f"2026-07-01 {13 + i}:00"))]

    ax1.scatter(
        outlier.lon,
        outlier.lat,
        s=150,
        facecolors="none",
        edgecolors="red",
        linewidth=2,
        zorder=10,
    )

    ax2.scatter(
        outlier.date,
        outlier.sst,
        s=150,
        facecolors="none",
        edgecolors="red",
        linewidth=2,
        zorder=10,
    )

    i += 1

ax1.set_xlabel("Longitude (°)")
ax1.set_ylabel("Latitude (°)")
ax1.set_title("Buddy observations")
ax1.legend(loc="best")

ax2.set_xlabel("Time")
ax2.set_ylabel("SST (°C)")
ax2.set_title("Sea surface temperature")
ax2.tick_params(axis="x", rotation=30)

plt.show()

In [ ]:
climatology_data = get_climatology_data()
climatology_data

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 8),
    constrained_layout=True,
)

plots = [
    ("sst", "sea-surface temperature", "RdYlBu_r"),
    ("sst_stdev1", "stdev1", "viridis"),
    ("sst_stdev2", "stdev2", "viridis"),
    ("sst_stdev3", "stdev3", "viridis"),
]

for ax, (var, title, cmap) in zip(axes.flat, plots, strict=False):
    climatology_data[var].isel(time=0).plot(
        ax=ax,
        cmap=cmap,
        add_colorbar=True,
    )
    ax.set_title(title)

plt.show()

In [ ]:
sst_stdev1 = Climatology(climatology_data.sst_stdev1)
sst_stdev2 = Climatology(climatology_data.sst_stdev2)
sst_stdev3 = Climatology(climatology_data.sst_stdev3)

In [ ]:
qc_buddy = do_bayesian_buddy_check(
    value=data.sst,
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    climatology=climatology_data.sst,
    stdev1=sst_stdev1,
    stdev2=sst_stdev2,
    stdev3=sst_stdev3,
    prior_probability_of_gross_error=0.05,
    quantization_interval=0.1,
    one_sigma_measurement_uncertainty=1.0,
    limits=[2, 2, 4],
    noise_scaling=3.0,
    maximum_anomaly=8.0,
    fail_probability=0.3,
)
pd.DataFrame({**data, "qc_flag": qc_buddy})